# Stage1

In [30]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##1. 라이브러리 불러오기

In [31]:
import csv
import json
import os

import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, chi2   #######윤서영
from sklearn.metrics import f1_score, classification_report   #######윤서영 classification_report 추가
from sklearn.preprocessing import OneHotEncoder   #######윤서영
import lightgbm as lgb   #######윤서영
import numpy as np   #######윤서영
from sklearn.base import BaseEstimator, TransformerMixin
import re
import pandas as pd


# 예측 대상 14개 클래스 (Macro-F1 계산에 사용)
ALL_CLASSES = [
    "read_file", "grep_search", "list_directory", "glob_pattern",
    "edit_file", "write_file", "apply_patch",
    "run_bash", "run_tests", "lint_or_typecheck",
    "ask_user", "plan_task", "web_search", "respond_only",
]

##2. 데이터 불러오기



In [32]:
DATA_DIR = "/content/drive/MyDrive/nlp_competition/data" ### 임유미 수정

# train.jsonl: 한 줄 = 샘플 하나
samples = [json.loads(line)
           for line in open(os.path.join(DATA_DIR, "train.jsonl"), encoding="utf-8")
           if line.strip()]

# train_labels.csv: id -> action 매핑
labels = {row["id"]: row["action"]
          for row in csv.DictReader(open(os.path.join(DATA_DIR, "train_labels.csv"), encoding="utf-8"))}

# 입력 X = current_prompt, 정답 y = action
X = [s["current_prompt"] for s in samples]
y = [labels[s["id"]] for s in samples]
ids = [s["id"] for s in samples]

# id로 원본 샘플(session_meta 포함)을 바로 찾기 위한 딕셔너리
samples_by_id = {s["id"]: s for s in samples}


print("samples:", len(X), "| classes:", len(set(y)))

samples: 70000 | classes: 14


##3. 학습 / 검증 데이터 분할

In [33]:
X_train, X_val, y_train, y_val, train_ids, val_ids = train_test_split(
    X, y, ids, test_size=0.2, stratify=y, random_state=42,
)
print("train:", len(X_train), "| val:", len(X_val))

train: 56000 | val: 14000


In [34]:
# session meta 추가 연결
from sklearn.compose import ColumnTransformer

def extract_session_features(sample):
  s = samples_by_id[sample]

  meta = s.get("session_meta", {}) or {}
  ws = meta.get("workspace", {}) or {}
  lang_mix = ws.get("language_mix") or {}
  dom_lang, dom_ratio = max(lang_mix.items(), key=lambda kv: kv[1]) if lang_mix else ("none", 0.0)

  return {
    "language_pref": meta.get("language_pref", "unknown"),
    "last_ci_status": ws.get("last_ci_status", "none"),
    "git_dirty": str(ws.get("git_dirty", False)),

    "user_tier": meta.get("user_tier", "unknown"),
    "turn_index": meta.get("turn_index", 0),
    "budget_tokens_remaining_log": np.log1p(meta.get("budget_tokens_remaining", 0)),
    "loc_log": np.log1p(ws.get("loc", 0)),
    "n_open_files": len(ws.get("open_files") or []),
    "dominant_code_lang": dom_lang,
    "dominant_code_ratio": dom_ratio,
    "n_code_langs": len(lang_mix),
    "elapsed_session_sec_log": np.log1p(meta.get("elapsed_session_sec", 0)),
}

# train_ids / val_ids 순서 그대로 매핑
session_train_df = pd.DataFrame([extract_session_features(i) for i in train_ids])
session_val_df = pd.DataFrame([extract_session_features(i) for i in val_ids])

# 원-핫 인코딩 (학습 데이터로 fit, 검증은 transform만)
CATEGORICAL_COLS = ["language_pref", "last_ci_status", "git_dirty", "user_tier", "dominant_code_lang"]
NUMERIC_COLS = ["turn_index", "budget_tokens_remaining_log", "loc_log", "n_open_files", "dominant_code_ratio", "n_code_langs", "elapsed_session_sec_log"]

session_encoder = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), CATEGORICAL_COLS),
    ("num", "passthrough", NUMERIC_COLS),   # 숫자는 변환 없이 그대로 통과
])

session_train_enc = session_encoder.fit_transform(session_train_df)
session_val_enc = session_encoder.transform(session_val_df)

print("생성된 피처:", session_encoder.get_feature_names_out())
print("shape:", session_train_enc.shape)


생성된 피처: ['cat__language_pref_en' 'cat__language_pref_ko'
 'cat__language_pref_mixed' 'cat__last_ci_status_failed'
 'cat__last_ci_status_none' 'cat__last_ci_status_passed'
 'cat__git_dirty_False' 'cat__git_dirty_True' 'cat__user_tier_enterprise'
 'cat__user_tier_free' 'cat__user_tier_pro' 'cat__dominant_code_lang_go'
 'cat__dominant_code_lang_java' 'cat__dominant_code_lang_py'
 'cat__dominant_code_lang_rs' 'cat__dominant_code_lang_ts'
 'cat__dominant_code_lang_tsx' 'cat__dominant_code_lang_vue'
 'cat__dominant_code_lang_yaml' 'num__turn_index'
 'num__budget_tokens_remaining_log' 'num__loc_log' 'num__n_open_files'
 'num__dominant_code_ratio' 'num__n_code_langs'
 'num__elapsed_session_sec_log']
shape: (56000, 26)


##4. 모델 정의와 학습 (TF-IDF + LightGBM)

In [35]:
#규칙 기반 피처 클래스
class RuleFeatureExtractor(BaseEstimator, TransformerMixin):

    def fit(self, X, y=None):
        return self   # 학습할 게 없으니 그냥 자기 자신 반환

    def transform(self, X):
        return np.array([self._extract(text) for text in X])

    def _extract(self, text):
        return [
            1 if re.search(r'\b(read|열어|읽어)\b', text, re.I) else 0,
            1 if re.search(r'(grep|검색|찾아).*(코드|텍스트|패턴)', text, re.I) else 0,
            1 if re.search(r'(폴더|디렉토리|directory).*(뭐|목록|구조)', text, re.I) else 0,
            1 if re.search(r'\*\.\w+|글롭|glob|확장자', text, re.I) else 0,
            1 if re.search(r'(수정|고쳐|바꿔|edit)', text, re.I) else 0,
            1 if re.search(r'(새로|생성|만들어|write).*(파일|코드)', text, re.I) else 0,
            1 if re.search(r'(diff|patch|패치|변경사항)', text, re.I) else 0,
            1 if re.search(r'(셸|shell|bash|명령어|터미널)', text, re.I) else 0,
            1 if re.search(r'(테스트|test).*(실행|돌려|run)', text, re.I) else 0,
            1 if re.search(r'(린트|lint|타입체크|맞춤법)', text, re.I) else 0,
            1 if '?' in text else 0,
            1 if re.search(r'(계획|설계|plan)', text, re.I) else 0,
            1 if re.search(r'(웹|인터넷|구글|web)', text, re.I) else 0,
            len(text),
            len(re.findall(r'[a-zA-Z]', text)),
        ]

rule_extractor = RuleFeatureExtractor()
rule_features_train = rule_extractor.transform(X_train)
rule_features_val = rule_extractor.transform(X_val)

In [36]:
#LightGBM의 성능평가지표를 Macro-F1 직접 만들기

def macro_f1_eval(y_true, y_pred_proba):
    y_pred = y_pred_proba.reshape(len(y_true), -1).argmax(axis=1)
    f1 = f1_score(y_true, y_pred, average="macro")
    return "macro_f1", f1, True   # True = "높을수록 좋음"

In [37]:
from scipy.sparse import hstack, csr_matrix

#1. TF-IDF
tfidf_chi2 = Pipeline([
  #1-1) TF-IDF
  ("tfidf", TfidfVectorizer(ngram_range=(1,3), min_df=2, max_features=80_000,
                         sublinear_tf=True, lowercase=True)),
  #1-2) chi2 피처 선택
  ("chi2", SelectKBest(chi2, k=30_000)),
])

# 가지2: 규칙 기반 피처
rule = RuleFeatureExtractor()

# 두 갈래를 FeatureUnion으로 결합
vectorizer_pipe = FeatureUnion([
    ("tfidf_chi2", tfidf_chi2),
    ("rule_features", rule),
])

vectorizer = vectorizer_pipe
X_train_vec = vectorizer.fit_transform(X_train, y_train)
X_val_vec = vectorizer.transform(X_val)

#session meta 피쳐 결합
X_train_final = hstack([X_train_vec, session_train_enc])
X_val_final = hstack([X_val_vec, session_val_enc])


#2. LightGBM
"""
param_grid = {
    "n_estimators": [300, 400],
    "num_leaves": [31, 50],
    "min_child_samples": [10, 20],
    "learning_rate": [0.05, 0.1],
}

clf = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", n_jobs=1, verbosity=1,random_state=42)
grid = RandomizedSearchCV(clf, param_grid, n_iter=8, scoring="f1_macro", cv=3, n_jobs=-1, verbose=2, random_state=42)
grid.fit(X_train_vec, y_train)

print("최적 파라미터:", grid.best_params_)"""

'\nparam_grid = {\n    "n_estimators": [300, 400],\n    "num_leaves": [31, 50],\n    "min_child_samples": [10, 20],\n    "learning_rate": [0.05, 0.1],\n}\n\nclf = lgb.LGBMClassifier(objective="multiclass", class_weight="balanced", n_jobs=1, verbosity=1,random_state=42)\ngrid = RandomizedSearchCV(clf, param_grid, n_iter=8, scoring="f1_macro", cv=3, n_jobs=-1, verbose=2, random_state=42)\ngrid.fit(X_train_vec, y_train)\n\nprint("최적 파라미터:", grid.best_params_)'

In [ ]:
#best_clf = lgb.LGBMClassifier(**grid.best_params_, objective="multiclass", class_weight="balanced", random_state=42)
clf = lgb.LGBMClassifier(n_estimators=400, num_leaves=80, min_child_samples=10, learning_rate=0.1,
                               objective="multiclass", class_weight="balanced", random_state=42)
best_clf = clf

best_clf.fit(
    X_train_final, y_train,
    eval_set=[(X_val_final, y_val)],
    eval_metric=macro_f1_eval,
    callbacks=[lgb.early_stopping(stopping_rounds=150)],
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 9.885356 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 213132
[LightGBM] [Info] Number of data points in the train set: 56000, number of used features: 10477
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.639057
[LightGBM] [Info] Start training from score -2.6

In [ ]:
tfidf_count = X_train_vec.shape[1] - rule_features_train.shape[1]

feature_names = (
    [f"tfidf_{i}" for i in range(tfidf_count)] +
    [f"rule_{i}" for i in range(rule_features_train.shape[1])] +
    list(session_encoder.get_feature_names_out())
)

gain_importance = best_clf.booster_.feature_importance(importance_type="gain")
importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": gain_importance,
}).sort_values("importance", ascending=False)

# 세션메타 피처들의 순위 확인
print(importance_df[importance_df["feature"].str.startswith((
    "cat__last_ci_status", "cat__git_dirty", "cat__user_tier", "num__turn_index", "num__budget_tokens_remaining_log", 'num__loc_log', 'num__n_open_files',
    'cat__dominant_code_lang_go', 'cat__dominant_code_lang_java', 'cat__dominant_code_lang_py', 'cat__dominant_code_lang_rs',
    'cat__dominant_code_lang_ts', 'cat__dominant_code_lang_tsx', 'cat__dominant_code_lang_vue', 'cat__dominant_code_lang_yaml',
    'num__turn_index', 'num__budget_tokens_remaining_log', 'num__loc_log' 'num__n_open_files', 'num__dominant_code_ratio', 'num__n_code_langs', 'num__elapsed_session_sec_log'))])


In [ ]:
tfidf_probs_train = best_clf.predict_proba(X_train_final)   # 학습셋 확률 (가중치 튜닝용)
tfidf_probs_val = best_clf.predict_proba(X_val_final)       # 검증셋 확률

rule_only_clf = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42,
)
rule_only_clf.fit(rule_features_train, y_train)
rule_probs_train = rule_only_clf.predict_proba(rule_features_train)
rule_probs_val = rule_only_clf.predict_proba(rule_features_val)

In [ ]:
# 여러 가중치 조합을 실험해서 앙상블 최적점 찾기

y_val_arr = np.array(y_val)
threshold = 0.6   # 확정 기준 (기존에 쓰시던 값)
results = []
for w in [0.95, 0.9, 0.85, 0.8, 0.75, 0.7]:
    for th in [0.7, 0.75, 0.8]:
        ensemble_probs = w * tfidf_probs_val + (1 - w) * rule_probs_val
        max_conf = ensemble_probs.max(axis=1)
        pred_idx = ensemble_probs.argmax(axis=1)
        pred_labels = np.array([best_clf.classes_[i] for i in pred_idx])

        macro_f1 = f1_score(y_val, pred_labels, labels=ALL_CLASSES, average="macro", zero_division=0)

        confirmed_mask = max_conf >= th
        if confirmed_mask.sum() == 0:
            continue
        confirmed_acc = (pred_labels[confirmed_mask] == np.array(y_val)[confirmed_mask]).mean()

        results.append({
            "w": w, "threshold": th,
            "Macro-F1": round(macro_f1, 4),
            "확정샘플수": confirmed_mask.sum(),
            "확정정확도": round(confirmed_acc, 4),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
best_w = 0.8
best_threshold = 0.8

def route_by_confidence(ids, tfidf_probs, rule_probs, classes, w=best_w, threshold=best_threshold): ### 임유미 수정 (인자로 classes를 받도록 함수 정의부 수정)
    """
    Stage1 앙상블 예측 확률을 보고, 확신도에 따라 라우팅하는 함수

    Parameters
    ----------
    ids : list
        각 샘플의 고유 id
    tfidf_probs : np.ndarray (샘플수, 14)
        TF-IDF+LightGBM 모델의 확률
    rule_probs : np.ndarray (샘플수, 14)
        규칙피처+LogReg 모델의 확률
    classes: probs의 각 컬럼이 어떤 클래스를 의미하는지 알려주는 배열
        (반드시 predict_proba를 만든 모델의 .classes_ 를 그대로 넘길 것) ### 임유미 추가
    w : float
        TF-IDF 쪽 가중치 (0.6 확정)
    threshold : float
        이 값 이상이면 Stage1에서 바로 확정

    Returns
    -------
    stage1_results : dict {id: action}
        확신도 높아서 Stage1에서 바로 결정된 것들
    stage2_input_ids : list
        확신도 낮아서 Stage2로 넘겨야 하는 id 목록
    stage2_probs : np.ndarray
        Stage2로 넘어가는 샘플들의 Stage1 확률 분포 (best_clf.predict_proba(X_vec) 결과)
    """
    ensemble_probs = w * tfidf_probs + (1 - w) * rule_probs
    max_conf = ensemble_probs.max(axis=1)      # 각 샘플의 "가장 높은 확률"
    pred_idx = ensemble_probs.argmax(axis=1)   # 그 확률에 해당하는 클래스 인덱스

    stage1_results = {}
    stage2_input_ids = []
    stage2_probs_list = []

    for sample_id, conf, idx, prob_row in zip(ids, max_conf, pred_idx, ensemble_probs):
        if conf >= threshold:
            #확신도 높음 → 확정
            stage1_results[sample_id] = classes[idx]   ### 임유미: ALL_CLASSES → classes로 변경
        else:
            #확신도 낮음 → Stage2로
            stage2_input_ids.append(sample_id)
            stage2_probs_list.append(prob_row)

    stage2_probs = np.array(stage2_probs_list) if stage2_probs_list else np.empty((0, len(ALL_CLASSES)))

    return stage1_results, stage2_input_ids, stage2_probs


In [ ]:

#stage2 들어갈 데이터 받는 예시 코드
stage1_results, stage2_input_ids, stage2_probs = route_by_confidence(
    ids=val_ids,
    tfidf_probs=tfidf_probs_val,
    rule_probs=rule_probs_val,
    classes=best_clf.classes_,   ### 임유미 추가
)

print(f"Stage1에서 확정된 샘플 수: {len(stage1_results)}")
print(f"Stage2로 넘어가는 샘플 수: {len(stage2_input_ids)}")

In [ ]:
# 확정된 샘플들(Stage1에서 끝난 것들)만 따로 정확도 확인
"""stage1_conf_probs = best_clf.predict_proba(X_val_vec)
max_conf = stage1_conf_probs.max(axis=1)
pred = best_clf.predict(X_val_vec)"""
ensemble_probs_val = best_w * tfidf_probs_val + (1-best_w) * rule_probs_val
max_conf = ensemble_probs_val.max(axis=1)
pred_idx = ensemble_probs_val.argmax(axis=1)
pred_labels = np.array([best_clf.classes_[i] for i in pred_idx])

confirmed_mask = max_conf >= best_threshold

confirmed_acc = (pred_labels[confirmed_mask] == np.array(y_val)[confirmed_mask]).mean()
print(f"✅ 확정 샘플 수: {confirmed_mask.sum()}")
print(f"🎯 확정 샘플 중 정확도: {confirmed_acc:.4f}")

##5. Stage1 검증 (Macro-F1)

In [ ]:
ensemble_pred = np.array([best_clf.classes_[i] for i in ensemble_probs_val.argmax(axis=1)])
macro_f1 = f1_score(y_val, ensemble_pred, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"Validation Macro-F1: {macro_f1:.4f}")

클래스별 F1점수 보기

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_val, ensemble_pred, labels=ALL_CLASSES, zero_division=0))

# Stage2

##1. 라이브러리 정의

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

## 2. Stage2 데이터셋 구성

In [ ]:
def build_stage2_input_text(sample, history_turns=3):
    history = sample.get("history", []) or []
    hist_text = "\n".join(str(turn) for turn in history[-history_turns:])
    return f"{hist_text}\n{sample.get('current_prompt', '')}".strip()

def build_stage2_dataset(stage2_ids, stage2_probs, samples_by_id, labels, session_encoder, history_turns=3):
    """
    stage2_ids/stage2_probs: route_by_confidence가 넘긴 것 그대로 (stage2_probs는 앙상블 확률)
    session_encoder: Stage1에서 이미 fit된 ColumnTransformer (train 기준) — transform만 함
    """
    texts, y_stage2 = [], []
    for sid in stage2_ids:
        s = samples_by_id[sid]
        texts.append(build_stage2_input_text(s, history_turns))
        y_stage2.append(labels[sid])

    # ---- session_meta 인코딩 (Stage1의 extract_session_features 재사용) ----
    session_df = pd.DataFrame([extract_session_features(sid) for sid in stage2_ids])
    session_enc = session_encoder.transform(session_df)
    if hasattr(session_enc, "toarray"):   # ColumnTransformer 결과가 sparse일 수 있음
        session_enc = session_enc.toarray()

    return texts, y_stage2, np.array(stage2_probs), session_enc.astype(np.float32)

X_stage2_text, y_stage2, stage1_probs_stage2, session_stage2_enc = build_stage2_dataset(
    stage2_input_ids, stage2_probs, samples_by_id, labels, session_encoder,
)
print(f"Stage2 학습 후보: {len(X_stage2_text)}건")
print(f"session_meta 인코딩 shape: {session_stage2_enc.shape}")
print(pd.Series(y_stage2).value_counts())

print("session_stage2_enc shape:", session_stage2_enc.shape)  # (샘플수, 26)이어야 함
print(session_encoder.get_feature_names_out())  # 'num__elapsed_session_sec_log' 포함 확인

##3. 분류 헤드 (K-fold로 stage2_input_ids 전체 예측 만들기)

In [ ]:
from sklearn.model_selection import StratifiedKFold

n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

y_stage2_arr = np.array(y_stage2)
oof_preds = np.empty(len(y_stage2_arr), dtype=object)  # out-of-fold 예측 저장용

embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
emb_all = embedder.encode(X_stage2_text, show_progress_bar=True)  # 전체 텍스트 한 번에 임베딩

feat_all = np.hstack([emb_all, stage1_probs_stage2, session_stage2_enc])

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(feat_all, y_stage2_arr)):
    head_fold = LogisticRegression(max_iter=1000, class_weight="balanced")
    head_fold.fit(feat_all[train_idx], y_stage2_arr[train_idx])
    oof_preds[val_idx] = head_fold.predict(feat_all[val_idx])
    print(f"Fold {fold_idx+1}/{n_splits} 완료")

# 이제 stage2_input_ids 전체에 대해 예측이 다 채워짐
stage2_oof_dict = dict(zip(stage2_input_ids, oof_preds))

##4. Stage2 Macro-F1 측정

In [ ]:
pred_s2_val = head.predict(feat_val)
stage2_macro_f1 = f1_score(y_s2_val, pred_s2_val, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"Stage2 단독 Validation Macro-F1: {stage2_macro_f1:.4f}")
print(classification_report(y_s2_val, pred_s2_val, labels=ALL_CLASSES, zero_division=0))

#Stage1+Stage2 전체 Macro-F1 측정

In [ ]:
##5. Stage1 + Stage2 통합 Macro-F1 (전체 val_ids 커버)

final_preds = dict(stage1_results)          # stage1 확정분 1,643건
final_preds.update(stage2_oof_dict)         # stage2 전체 12,357건 (OOF 예측)

final_ids = list(final_preds.keys())
print(f"통합 샘플 수: {len(final_ids)}")    # 이제 14,000이 나와야 정상

y_true_final = [labels[i] for i in final_ids]
y_pred_final = [final_preds[i] for i in final_ids]

overall_macro_f1 = f1_score(y_true_final, y_pred_final, labels=ALL_CLASSES, average="macro", zero_division=0)
print(f"🎯 Stage1+Stage2 통합 Validation Macro-F1: {overall_macro_f1:.4f}")
print(classification_report(y_true_final, y_pred_final, labels=ALL_CLASSES, zero_division=0))

#전체 데이터로 재학습 & 모델 저장

In [ ]:
# 전체 학습 데이터로 재학습
vectorizer_final = vectorizer_pipe
X_full_vec = vectorizer_final.fit_transform(X, y)

clf_final = clf
clf_final.fit(X_full_vec, y)

rule_features_full = rule_extractor.transform(X)
rule_only_clf_final = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
rule_only_clf_final.fit(rule_features_full, y)

In [ ]:
# 저장 (추론용 script.py가 ./model/tfidf_lgbm.pkl 을 불러옵니다)
os.makedirs("./model", exist_ok=True)

joblib.dump(vectorizer_final, "./model/vectorizer.pkl", compress=3)
joblib.dump(clf_final, "./model/tfidf_lgbm.pkl", compress=3)
joblib.dump(rule_only_clf_final, "./model/rule_logreg.pkl", compress=3)

print("저장 완료: ./model/tfidf_lgbm.pkl")